# Testing Word Ops Comparison Algorithm

In [1]:
import sys
import os
sys.path.append(os.path.abspath('../../'))  # Add root to path to import algo

from algo.evaluate import evaluate_comparisons
import json

### Функция, которая переводит список WordOp в json 

In [19]:
from __future__ import annotations

import json
from typing import Any, Dict, List, Sequence, Union

from algo.structures import WordOp


def word_ops_to_evaluate_input(
    ops: Sequence[WordOp],
    *,
    as_json_string: bool = False,
) -> Union[List[Dict[str, Any]], str]:
    """
    Список WordOp → данные для evaluate_comparisons (список dict).
    По умолчанию возвращает list[dict] — его можно сразу передать как первый или второй аргумент.
    Если as_json_string=True — JSON-строка (для записи в файл); перед вызовом evaluate_comparisons сделайте json.loads.
    """
    out: List[Dict[str, Any]] = []
    for op in ops:
        ot = op.op_type.value if hasattr(op.op_type, "value") else str(op.op_type)
        out.append(
            {
                "op_type": ot,
                "base_token": op.base_token,
                "other_token": op.other_token,
            }
        )
    if as_json_string:
        return json.dumps(out, ensure_ascii=False)
    return out

In [29]:
import sys
import os
import docx
import re
# Добавляем корневую папку проекта в путь, чтобы импортировать algo
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '../../')))

from algo.comparison import compare_tokens
from algo.structures import Weights

def read_docx(path):
    doc = docx.Document(path)
    return " ".join([p.text for p in doc.paragraphs])

def tokenize(text):
    return re.findall(r'\S+', text)

# 1. Читаем тексты
text_m10 = read_docx('data/clean/M10.docx')
text_p81 = read_docx('data/clean/МИД181.docx')

# 2. Токенизируем
tokens_m10 = tokenize(text_m10)
tokens_p81 = tokenize(text_p81)

# 3. Сравниваем (можно выбрать "myers", "levenshtein", "levenshtein_nested")
test_values = compare_tokens(tokens_m10, tokens_p81, Weights(1, 1, 1, 1, 1, 1), algorithm="levenshtein_v2")


In [30]:
print(word_ops_to_evaluate_input(test_values))

[{'op_type': 'delete', 'base_token': 'възвеселе', 'other_token': None}, {'op_type': 'delete', 'base_token': 'высоты', 'other_token': None}, {'op_type': 'delete', 'base_token': 'ѹтвержⷣенїа', 'other_token': None}, {'op_type': 'delete', 'base_token': 'оцѣщеⷩ\ue003а', 'other_token': None}, {'op_type': 'delete', 'base_token': 'зрⷦ\ue003', 'other_token': None}, {'op_type': 'delete', 'base_token': 'нб҃сї', 'other_token': None}, {'op_type': 'delete', 'base_token': 'вїдѣнїеⷨ', 'other_token': None}, {'op_type': 'delete', 'base_token': 'сл҃вы', 'other_token': None}, {'op_type': 'delete', 'base_token': 'сл҃нце', 'other_token': None}, {'op_type': 'delete', 'base_token': 'въ', 'other_token': None}, {'op_type': 'delete', 'base_token': 'ѧвлеⷩ\ue003е', 'other_token': None}, {'op_type': 'delete', 'base_token': 'проповѣдаѧ', 'other_token': None}, {'op_type': 'delete', 'base_token': 'въ', 'other_token': None}, {'op_type': 'delete', 'base_token': 'ісходѣ', 'other_token': None}, {'op_type': 'equal', 'base_

In [31]:
import json

with open('/Users/ehlamkin/Desktop/M10_MID181_correct.json', 'r', encoding='utf-8') as f:
    m10_mid181_correct = json.load(f)


print(m10_mid181_correct)

[{'id': 20290, 'op_type': 'delete', 'base_index': 0, 'other_index': None, 'base_token': 'възвеселе', 'other_token': None, 'base_paragraph_index': 0, 'char_ops': []}, {'id': 20291, 'op_type': 'delete', 'base_index': 1, 'other_index': None, 'base_token': 'высоты', 'other_token': None, 'base_paragraph_index': 0, 'char_ops': []}, {'id': 20292, 'op_type': 'delete', 'base_index': 2, 'other_index': None, 'base_token': 'ѹтвержⷣенїа', 'other_token': None, 'base_paragraph_index': 0, 'char_ops': []}, {'id': 20293, 'op_type': 'delete', 'base_index': 3, 'other_index': None, 'base_token': 'оцѣщеⷩ\ue003а', 'other_token': None, 'base_paragraph_index': 0, 'char_ops': []}, {'id': 20294, 'op_type': 'delete', 'base_index': 4, 'other_index': None, 'base_token': 'зрⷦ\ue003', 'other_token': None, 'base_paragraph_index': 0, 'char_ops': []}, {'id': 20295, 'op_type': 'delete', 'base_index': 5, 'other_index': None, 'base_token': 'нб҃сї', 'other_token': None, 'base_paragraph_index': 0, 'char_ops': []}, {'id': 202

In [32]:
results = evaluate_comparisons(word_ops_to_evaluate_input(test_values), m10_mid181_correct)

print("Recall:", results['recall'])
print("Precision:", results['precision'])
print("F1 Score:", results['f1_score'])
print("\n--- Recall Elements ---")
for el in results['recall_elements']:
    print(el)
    
print("\n--- Precision Elements ---")
for el in results['precision_elements']:
    print(el)

Recall: 0.979002624671916
Precision: 0.9588688946015425
F1 Score: 0.9688311688311689

--- Recall Elements ---
{'op1': 'delete', 'op2': 'delete', 'other_token1': None, 'other_token2': None, 'base_token': 'възвеселе'}
{'op1': 'delete', 'op2': 'delete', 'other_token1': None, 'other_token2': None, 'base_token': 'высоты'}
{'op1': 'delete', 'op2': 'delete', 'other_token1': None, 'other_token2': None, 'base_token': 'ѹтвержⷣенїа'}
{'op1': 'delete', 'op2': 'delete', 'other_token1': None, 'other_token2': None, 'base_token': 'оцѣщеⷩ\ue003а'}
{'op1': 'delete', 'op2': 'delete', 'other_token1': None, 'other_token2': None, 'base_token': 'зрⷦ\ue003'}
{'op1': 'delete', 'op2': 'delete', 'other_token1': None, 'other_token2': None, 'base_token': 'нб҃сї'}
{'op1': 'delete', 'op2': 'delete', 'other_token1': None, 'other_token2': None, 'base_token': 'вїдѣнїеⷨ'}
{'op1': 'delete', 'op2': 'delete', 'other_token1': None, 'other_token2': None, 'base_token': 'сл҃вы'}
{'op1': 'delete', 'op2': 'delete', 'other_token1

In [3]:
import json

# Открываем файл в режиме чтения
with open('data2/98.json', 'r', encoding='utf-8') as f:
    kanon = json.load(f)

with open('data2/100.json', 'r', encoding='utf-8') as f:
    algo = json.load(f)

# Теперь переменная 'data' содержит данные из файла
print(kanon)
print(algo)

[{'id': 12592, 'op_type': 'replace', 'base_index': 0, 'other_index': 0, 'base_token': 'блуд', 'other_token': 'блѫⷣ', 'base_paragraph_index': 0, 'char_ops': [{'id': 12732, 'op_type': 'change', 'from_str': 'у', 'to_str': 'ѫ', 'class_id': None}, {'id': 12733, 'op_type': 'change', 'from_str': 'д', 'to_str': 'ⷣ', 'class_id': None}]}, {'id': 12593, 'op_type': 'replace', 'base_index': 1, 'other_index': 1, 'base_token': 'женескъ', 'other_token': 'женскыи', 'base_paragraph_index': 0, 'char_ops': [{'id': 12734, 'op_type': 'delete', 'from_str': 'е', 'to_str': '', 'class_id': None}, {'id': 12735, 'op_type': 'change', 'from_str': 'ъ', 'to_str': 'ы', 'class_id': None}, {'id': 12736, 'op_type': 'insert', 'from_str': '', 'to_str': 'и', 'class_id': None}]}, {'id': 12594, 'op_type': 'replace', 'base_index': 2, 'other_index': 2, 'base_token': 'поглумленїе', 'other_token': 'проглоумленїе', 'base_paragraph_index': 0, 'char_ops': [{'id': 12737, 'op_type': 'insert', 'from_str': '', 'to_str': 'р', 'class_id':

In [4]:
# Let's create mock comparisons
# Assume the base text is: "The quick brown fox jumps"
# Canonical Other Text 1: "The fast brown fox leaps"
# Prediction Other Text 2: "The quick red fox jumps high"

comp1 = [
    {'op_type': 'equal', 'base_token': 'The', 'other_token': 'The'},
    {'op_type': 'replace', 'base_token': 'quick', 'other_token': 'fast'},
    {'op_type': 'equal', 'base_token': 'brown', 'other_token': 'brown'},
    {'op_type': 'equal', 'base_token': 'fox', 'other_token': 'fox'},
    {'op_type': 'replace', 'base_token': 'jumps', 'other_token': 'leaps'}
]

comp2 = [
    {'op_type': 'equal', 'base_token': 'The', 'other_token': 'The'},
    {'op_type': 'equal', 'base_token': 'quick', 'other_token': 'quick'},
    {'op_type': 'replace', 'base_token': 'brown', 'other_token': 'red'},
    {'op_type': 'equal', 'base_token': 'fox', 'other_token': 'fox'},
    {'op_type': 'equal', 'base_token': 'jumps', 'other_token': 'jumps'},
    {'op_type': 'insert', 'base_token': None, 'other_token': 'high'}
]


In [5]:
results = evaluate_comparisons(kanon, algo)

print("Recall:", results['recall'])
print("Precision:", results['precision'])
print("F1 Score:", results['f1_score'])
print("\n--- Recall Elements ---")
for el in results['recall_elements']:
    print(el)
    
print("\n--- Precision Elements ---")
for el in results['precision_elements']:
    print(el)

Recall: 0.9547141796585004
Precision: 0.9477172312223858
F1 Score: 0.9512028384095641

--- Recall Elements ---
{'op1': 'replace', 'op2': 'replace', 'other_token1': 'блѫⷣ', 'other_token2': 'блѫⷣ', 'base_token': 'блуд'}
{'op1': 'replace', 'op2': 'replace', 'other_token1': 'женскыи', 'other_token2': 'женскыи', 'base_token': 'женескъ'}
{'op1': 'replace', 'op2': 'replace', 'other_token1': 'проглоумленїе', 'other_token2': 'проглоумленїе', 'base_token': 'поглумленїе'}
{'op1': 'replace', 'op2': 'replace', 'other_token1': 'оч͡има,', 'other_token2': 'оч͡има,', 'base_token': 'очїма'}
{'op1': 'equal', 'op2': 'equal', 'other_token1': 'и', 'other_token2': 'и', 'base_token': 'и'}
{'op1': 'replace', 'op2': 'replace', 'other_token1': 'бръвма', 'other_token2': 'бръвма', 'base_token': 'бровма'}
{'op1': 'equal', 'op2': 'equal', 'other_token1': 'еѧ', 'other_token2': 'еѧ', 'base_token': 'еѧ'}
{'op1': 'equal', 'op2': 'equal', 'other_token1': 'познана', 'other_token2': 'познана', 'base_token': 'познана'}
{'op

In [6]:
# Example 2: Test inserts and deletes
comp3 = [
    {'op_type': 'equal', 'base_token': 'A', 'other_token': 'A'},
    {'op_type': 'insert', 'base_token': None, 'other_token': 'new'},
    {'op_type': 'equal', 'base_token': 'test', 'other_token': 'test'},
    {'op_type': 'delete', 'base_token': 'word', 'other_token': None}
]

comp4 = [
    {'op_type': 'equal', 'base_token': 'A', 'other_token': 'A'},
    {'op_type': 'insert', 'base_token': None, 'other_token': 'new'},
    {'op_type': 'equal', 'base_token': 'test', 'other_token': 'test'},
    {'op_type': 'delete', 'base_token': 'word', 'other_token': None}
]

res_perfect = evaluate_comparisons(comp3, comp4)
print("Perfect Match Recall:", res_perfect['recall']) # Should be 1.0

Perfect Match Recall: 1.0
